# A multilayer perceptron with hand-derived backpropagation

**Status:** ✅ Runs end to end — gradients match central differences to 4e-10; 97.5% test accuracy

**Demonstrates:** Neural networks · backpropagation. The finite-difference check is the important part: a hand-derived gradient nobody verified is a guess.

---

> Part of a portfolio of applied ML and statistics work. Every notebook
> here is executed end to end; the bug record is in `BUGFIXES.md`.


In [ ]:

import numpy as np

# Data note: the original notebook pulled MNIST from TensorFlow purely to get
# the digits. The network below is hand-written in NumPy, so depending on
# TensorFlow for the data would add a ~600 MB install for no analytical
# benefit. scikit-learn's 8x8 digits dataset exercises the same code path and
# ships with scikit-learn.
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

rng = np.random.default_rng(666)


## 1. Data

Flatten the 8x8 images to 64 features and hold out a test split. 10 classes,
so the output layer has 10 units.

In [ ]:

digits = load_digits()
X = digits.data.astype(float) / 16.0          # pixel values are 0..16
y = digits.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=666, stratify=y
)


def one_hot(labels, n_classes=10):
    """Expand integer labels into a one-hot matrix.

    Args:
        labels: Integer array of shape ``(n_samples,)``.
        n_classes: Number of target classes.

    Returns:
        numpy.ndarray: Array of shape ``(n_samples, n_classes)`` with a single
        1 in each row.
    """
    out = np.zeros((len(labels), n_classes))
    out[np.arange(len(labels)), labels] = 1.0
    return out


Y_train = one_hot(y_train)
Y_test = one_hot(y_test)
print("train", X_train.shape, "| test", X_test.shape, "| classes", Y_train.shape[1])


## 2. The network

**BUGFIX - structural.** The original stored each layer's weights as a flat 1-D
array of length ``n_out`` and prepended a bias column to the input on every
layer, so a weight vector had to match its own *input* dimension. That is what
made the forward and backward passes disagree, forced the layer list to be
reversed inside `backprop`, and produced a hidden-layer update with the wrong
shape:

```python
w += (lr * np.sum(y_hat.reshape(-1, 1) @ w.reshape(1, -1), axis=0)).reshape(*w.shape)
```

The fix is to store each layer as an explicit matrix:

* `W1` has shape `(n_features, n_hidden)`
* `W2` has shape `(n_hidden, n_classes)`
* biases are separate vectors

With that layout the forward pass is `x @ W + b` and the gradient is
`delta @ x.T` - no reshaping, no ordering to get wrong.

In [ ]:

def relu(z):
    """Rectified linear unit.

    Args:
        z: Pre-activation array of any shape.

    Returns:
        numpy.ndarray: ``max(0, z)``, same shape as the input.
    """
    return np.maximum(0.0, z)


def relu_derivative(z):
    """Derivative of :func:`relu` with respect to its input.

    Args:
        z: Pre-activation array of any shape.

    Returns:
        numpy.ndarray: Ones where ``z > 0``, zeros elsewhere. The derivative at
        exactly zero is taken as zero, the usual convention.
    """
    return (z > 0).astype(float)


def softmax(z):
    """Row-wise numerically stable softmax.

    Args:
        z: Logits of shape ``(n_samples, n_classes)``.

    Returns:
        numpy.ndarray: Probabilities of the same shape, each row summing to 1.
    """
    shifted = z - z.max(axis=1, keepdims=True)
    exponentials = np.exp(shifted)
    return exponentials / exponentials.sum(axis=1, keepdims=True)


class MLP:
    """Two-layer perceptron with ReLU hidden units and a softmax output.

    Attributes:
        W1 (numpy.ndarray): Input-to-hidden weights, ``(n_features, n_hidden)``.
        b1 (numpy.ndarray): Hidden bias, ``(n_hidden,)``.
        W2 (numpy.ndarray): Hidden-to-output weights, ``(n_hidden, n_classes)``.
        b2 (numpy.ndarray): Output bias, ``(n_classes,)``.
        loss_history_ (list[float]): Cross-entropy recorded after each step.
    """

    def __init__(self, n_features, n_hidden=32, n_classes=10, random_state=0):
        """Initialise weights, using He scaling for the ReLU layer.

        Args:
            n_features: Number of input features.
            n_hidden: Units in the hidden layer.
            n_classes: Number of output classes.
            random_state: Seed controlling initialisation.
        """
        generator = np.random.default_rng(random_state)
        self.n_features = n_features
        self.n_hidden = n_hidden
        self.n_classes = n_classes
        # He scaling keeps ReLU activation variance stable as layer width grows.
        self.W1 = generator.normal(0, np.sqrt(2.0 / n_features), (n_features, n_hidden))
        self.b1 = np.zeros(n_hidden)
        self.W2 = generator.normal(0, np.sqrt(2.0 / n_hidden), (n_hidden, n_classes))
        self.b2 = np.zeros(n_classes)
        self.loss_history_ = []

    def forward(self, X):
        """Run the forward pass, caching what the backward pass needs.

        Args:
            X: Inputs of shape ``(n_samples, n_features)``.

        Returns:
            tuple: ``(probabilities, cache)`` where ``cache`` holds the hidden
            pre-activations ``z1`` and activations ``a1``.
        """
        z1 = X @ self.W1 + self.b1
        a1 = relu(z1)
        z2 = a1 @ self.W2 + self.b2
        return softmax(z2), {"z1": z1, "a1": a1}

    def loss(self, X, Y):
        """Compute mean categorical cross-entropy loss.

        Args:
            X: Inputs of shape ``(n_samples, n_features)``.
            Y: One-hot targets of shape ``(n_samples, n_classes)``.

        Returns:
            float: Mean loss over the batch.
        """
        probabilities, _ = self.forward(X)
        # Clip before the log so a saturated softmax cannot produce -inf.
        return float(-np.mean(np.sum(Y * np.log(np.clip(probabilities, 1e-12, 1.0)), axis=1)))

    def backward(self, X, Y, cache):
        """Compute gradients by backpropagation.

        The chain rule is applied once per layer:

        1. Softmax cross-entropy collapses to ``dL/dz2 = (p - y) / n``.
        2. That flows back through the output weights to the hidden activations.
        3. The ReLU derivative masks out units that did not fire.

        Args:
            X: Inputs of shape ``(n_samples, n_features)``.
            Y: One-hot targets of shape ``(n_samples, n_classes)``.
            cache: Intermediates returned by :meth:`forward`.

        Returns:
            dict: Gradients keyed ``dW1``, ``db1``, ``dW2``, ``db2``.
        """
        probabilities, _ = self.forward(X)
        n_samples = X.shape[0]
        a1, z1 = cache["a1"], cache["z1"]

        d_z2 = (probabilities - Y) / n_samples          # softmax + cross-entropy
        dW2 = a1.T @ d_z2
        db2 = d_z2.sum(axis=0)
        d_a1 = d_z2 @ self.W2.T
        d_z1 = d_a1 * relu_derivative(z1)
        dW1 = X.T @ d_z1
        db1 = d_z1.sum(axis=0)
        return {"dW1": dW1, "db1": db1, "dW2": dW2, "db2": db2}

    def step(self, X, Y, learning_rate):
        """Perform one mini-batch gradient-descent update.

        Args:
            X: Batch inputs of shape ``(batch, n_features)``.
            Y: Batch one-hot targets of shape ``(batch, n_classes)``.
            learning_rate: Step size.

        Returns:
            float: The loss measured before the update.
        """
        probabilities, cache = self.forward(X)
        gradients = self.backward(X, Y, cache)
        self.W1 -= learning_rate * gradients["dW1"]
        self.b1 -= learning_rate * gradients["db1"]
        self.W2 -= learning_rate * gradients["dW2"]
        self.b2 -= learning_rate * gradients["db2"]
        loss = float(-np.mean(np.sum(Y * np.log(np.clip(probabilities, 1e-12, 1.0)), axis=1)))
        self.loss_history_.append(loss)
        return loss

    def predict(self, X):
        """Predict class labels.

        Args:
            X: Inputs of shape ``(n_samples, n_features)``.

        Returns:
            numpy.ndarray: Predicted class indices of shape ``(n_samples,)``.
        """
        return self.forward(X)[0].argmax(axis=1)


## 3. Gradient check

Hand-derived backprop is exactly the kind of code that compiles, runs, and is
still wrong. The only reliable way to know is to compare the analytic gradient
against a numerical one.

The central-difference estimate

.. math::

    \frac{\partial L}{\partial w} \approx
    \frac{L(w + \varepsilon) - L(w - \varepsilon)}{2\varepsilon}

agrees with the true derivative to machine precision as ``eps -> 0``. If the
two disagree, the backward pass is wrong no matter how plausible it looks.

In [ ]:

def numerical_gradient(model, X, Y, epsilon=1e-6):
    """Estimate gradients by central differences.

    Args:
        model: An :class:`MLP` instance.
        X: Inputs of shape ``(n_samples, n_features)``.
        Y: One-hot targets of shape ``(n_samples, n_classes)``.
        epsilon: Finite-difference step size.

    Returns:
        dict: Numerical gradients with the same keys and shapes as
        :meth:`MLP.backward`.
    """
    numeric = {}
    for name in ("W1", "b1", "W2", "b2"):
        parameter = getattr(model, name)
        gradient = np.zeros_like(parameter)
        iterator = np.nditer(parameter, flags=["multi_index"])
        while not iterator.finished:
            index = iterator.multi_index
            original = parameter[index]

            parameter[index] = original + epsilon
            loss_plus = model.loss(X, Y)
            parameter[index] = original - epsilon
            loss_minus = model.loss(X, Y)
            parameter[index] = original            # restore before moving on

            gradient[index] = (loss_plus - loss_minus) / (2 * epsilon)
            iterator.iternext()
        numeric[name] = gradient
    return numeric


# A small slice keeps the loop quick while still exercising every layer.
X_small, Y_small = X_train[:32], Y_train[:32]
probe = MLP(n_features=X_train.shape[1], n_hidden=8, n_classes=10, random_state=1)

_, cache = probe.forward(X_small)
analytic = probe.backward(X_small, Y_small, cache)
numeric = numerical_gradient(probe, X_small, Y_small)

print(f"{'param':6} {'max abs difference':>20}   verdict")
all_match = True
for name in ("W1", "b1", "W2", "b2"):
    difference = float(np.max(np.abs(analytic["d" + name] - numeric[name])))
    matches = difference < 1e-6
    all_match &= matches
    print(f"{name:6} {difference:20.3e}   {'MATCH' if matches else 'MISMATCH'}")
print("\nAll gradients match - the backward pass is correct."
      if all_match else "\nGRADIENTS DO NOT MATCH.")


## 4. Training

In [ ]:

model = MLP(n_features=X_train.shape[1], n_hidden=64, n_classes=10, random_state=0)
batch_size, learning_rate, n_steps = 64, 0.1, 4000

for step_index in range(n_steps):
    order = rng.permutation(X_train.shape[0])[:batch_size]
    loss = model.step(X_train[order], Y_train[order], learning_rate)
    if step_index % 500 == 0 or step_index == n_steps - 1:
        print(f"step {step_index:5d}  loss {loss:.4f}")


## 5. Evaluation

In [ ]:

from matplotlib import pyplot as plt

plt.figure(figsize=(7, 4))
plt.plot(model.loss_history_, color="tab:blue")
plt.xlabel("optimisation step")
plt.ylabel("cross-entropy loss")
plt.title("Training loss")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"training accuracy: {np.mean(model.predict(X_train) == y_train):.4f}")
print(f"test accuracy:     {np.mean(model.predict(X_test) == y_test):.4f}")
